In [1]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

smith_key = os.getenv("LANG_SMITH_API_KEY")

In [6]:
# setup vector database
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                  model_kwargs={"device": "cpu"})
vectore_store = Chroma(
    collection_name="cs50_policy",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
vectore_store.delete_collection()

In [7]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
# read txt file
with open("policy.txt",'r',encoding="utf-8") as file:
    text = file.read()

# chunk text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 400,
    chunk_overlap = 150,
    separators=[" ", ""]
)

chunks = text_splitter.split_text(text)

# convert chunks to Documents
documents = [Document(page_content=chunk) for chunk in chunks]


In [9]:
# store documents in chromaDB
vectore_store.add_documents(documents=documents)

['7b21f6e7-8d1a-4226-8ca9-4b4e5ee7e96b',
 '06ceb70c-c632-4bec-9384-ba119cb0affa',
 '59c4dc2b-1f00-42c0-ab2a-5ce7ce780c6c',
 '066841ea-d0f7-4edc-a57a-8b1360a22574',
 'e7908d99-e87e-4500-a18a-bf82bc6f7d09',
 '34908933-3bd2-4fee-8757-b8d98a5400ff',
 '0bceddf5-4c37-45c1-af32-5cb52d0f90f6',
 '8e26d64a-87fe-4e7d-a29c-e61378635ae6',
 'd9924a77-709f-47d6-88a7-a14b211dc2d8',
 '66bcb671-f801-4d52-8669-887a7c7db00f',
 'c4a36538-45c2-40d7-aef3-b3f752d07444',
 'ec104685-b306-441f-9511-d5007032a535',
 '8ae97b4c-8338-42c0-b0a4-fbd1f71a3aa1',
 '647f954d-f499-4b14-bc1d-b3891c42e455',
 'f9207084-afd9-4d82-8f90-f91157c36596',
 '0e13c6f7-6ad9-4bc9-b3bf-9f674a7d3eb0',
 'c9949853-ebcf-4c29-bba8-0b98598b053c',
 '0439bf8a-a327-4ee4-afcf-1b8dc87486b2',
 '808ed0f5-19cd-4072-a8cd-358fd6f5c42a',
 'b98c194a-a449-48e6-ba18-0e2a993999b9',
 'c1687511-2482-4d28-84d8-7eb6ca579998',
 '2641c387-816b-4b46-8a00-08a7b14b908b',
 '6b299daf-777d-4944-bffa-c580f1c4a4c2',
 'bd94d3bc-ea37-4a03-b6c0-44049315eccc',
 'b5574c94-44af-

In [50]:
# similarity search
results = vectore_store.similarity_search("can i take this course?",k=2)
results

[Document(id='82f896a5-2c13-4aec-a0b4-a3e836290b7c', metadata={}, page_content='examine our course map to determine if this class is the right one for you at this point in your development as a learner.\n\nCan this course be used to fulfill official academic requirements for my university, college, or school?\nOur free and verified certificates are not accredited academic'),
 Document(id='a51eae69-67f1-4567-9cbe-263d7d6c6ee7', metadata={}, page_content='in signing up for these free services.\n\nDoes this course have prerequisites?\nWhile you are not required to take or show proof that you have passed a previous course, it is highly recommended that you examine our course map to determine if this class is the right one for you at this point in your')]

In [45]:
vectore_store.delete_collection()

In [10]:
len(vectore_store.get()["ids"])

133

In [11]:
chunks

['Should I Take This Course?\nAm I or my child too young to take this course?\nOur courses are perhaps best suited for learners ages 12 and up. Younger learners might need a hand from a parent. Please note that your local laws and policies may prevent learners under certain ages from utilizing free third-party services associated with this course. Accordingly, younger learners may need your assistance',
 'under certain ages from utilizing free third-party services associated with this course. Accordingly, younger learners may need your assistance in signing up for these free services.\n\nDoes this course have prerequisites?\nWhile you are not required to take or show proof that you have passed a previous course, it is highly recommended that you examine our course map to determine if this class is',
 'to take or show proof that you have passed a previous course, it is highly recommended that you examine our course map to determine if this class is the right one for you at this point in

In [5]:
# hybrid search

In [6]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

In [7]:
vector_retriever = vectore_store.as_retriever(search_kwargs={"k":1})
bm_retriever = BM25Retriever.from_documents(documents)
bm_retriever.k = 1

In [8]:
retriever = EnsembleRetriever(retrievers=[vector_retriever,bm_retriever],weights=[0.3,0.7])

In [9]:
retriever.invoke("can i take this course?")

[Document(metadata={}, page_content='Should I Take This Course?\nAm I or my child too young to take this course?\nOur courses are perhaps best suited for learners ages 12 and up. Younger learners might need a hand from a parent. Please note that your local laws and policies may prevent learners under certain ages from utilizing free'),
 Document(id='82f896a5-2c13-4aec-a0b4-a3e836290b7c', metadata={}, page_content='examine our course map to determine if this class is the right one for you at this point in your development as a learner.\n\nCan this course be used to fulfill official academic requirements for my university, college, or school?\nOur free and verified certificates are not accredited academic')]

In [10]:
from langsmith import traceable

In [11]:
@traceable(name="retriever")
def retriever_test(query,retriever):
    return [doc.page_content for doc in retriever.invoke(query)]

In [12]:
retriever_test("can i take this course?",retriever)

['Should I Take This Course?\nAm I or my child too young to take this course?\nOur courses are perhaps best suited for learners ages 12 and up. Younger learners might need a hand from a parent. Please note that your local laws and policies may prevent learners under certain ages from utilizing free',
 'examine our course map to determine if this class is the right one for you at this point in your development as a learner.\n\nCan this course be used to fulfill official academic requirements for my university, college, or school?\nOur free and verified certificates are not accredited academic']